<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_pinn_thermal_prediction_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 — PINN Thermal Prediction

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L7.1 · Fundamentals of PINNs**

A winding in the slot of an electrical machine carries a current and heats up.
How hot does it get inside, where no thermometer can go? You compute the answer
three ways — by finite differences, and by a physics-informed network with its
walls imposed by soft and by hard enforcement — and score all three against the
exact solution, which this slot happens to have.

### The running example

* **The slot.** 10 × 20 mm, in the stator of an electrical machine, its walls
  held at the temperature of the cooled iron, 90 °C.
* **The winding.** 32 round copper wires of 2 mm, in 4 columns of 8, all
  carrying the same current, 10 to 45 A. Each wire makes $RI^2$ of heat, and
  hot copper makes more of it.
* **The bundle** is copper, enamel and resin together, with an effective
  conductivity of 0.70 W/m·K — which is why the inside runs hot.

### Sections

| | what you do | TODO |
|---|---|---|
| **1** | the slot, and the heat its winding makes | the heat source |
| **2** | the exact solution and the finite-difference solution, side by side | — |
| **3** | optimising the mesh density for an agreed accuracy of 0.01 K | the hot-spot error of each grid |
| **4** | a PINN: collocation points, network size, soft and hard enforcement | the residual |
| **5** | the three answers compared | — |
| **6** | your report | your answers |
| **7** | mini project proposal | — |

The exercise form has three TODO cells, each with its answer in the comment
directly above the line to write. The light form has every cell written out.
Numbers are printed with at most two decimals.

---

## 0 · Setup

**What these cells do.** Fetch the three library files and import them.

**How.** On Colab the first cell downloads `course_core.py`, `pinn_core.py` and `problem.py` from the public course repository; the second imports them, fixes the random seed and sets the folder the report is written to.

In [ ]:
# files-cell v2 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository on every run, so a file left
# over from an earlier session cannot shadow it. Run this cell first.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    # on Colab always re-fetch: /content outlives a session and a stale copy
    # would silently shadow an updated one. Locally the repo's own files stay.
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
# a module imported before this cell would keep the previous file's contents
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

cc.OUTPUT_DIR = "Ex07.1_outputs"                          # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

**What you should see.** `device: cpu` on most machines and `dtype: torch.float64`.
Double precision is deliberate: a second derivative of a network is a
difference of differences, and in single precision the residual would be
dominated by rounding. No GPU is needed anywhere in this set.

---

## 1 · The slot and its heat

**What this cell does.** Prints the slot, its winding and the heat the winding makes at rated current.

**How.** `describe_slot()` reads the constants in `problem.py` and works out the resistance of one wire, its $RI^2$, and the heat of the whole winding. The problem, in the operator form of L7.1's **Operator Form** slide, is

$$k_{\text{eff}}\,\nabla^{2}\theta + q(\theta, I) = 0 \quad \text{in the slot}, \qquad \theta = 0 \ \text{ on the four walls},$$

where $\theta$ is the temperature rise above the wall. Every parameter is known and only the field is missing: a **forward problem**.

> In L7.1 $\theta$ is the network's parameters; here it is the temperature rise, the usual heat-transfer letter. Which is meant is always clear from what it is attached to.

In [ ]:
pb.describe_slot()

**What you should see.**

```
  slot            : 10 x 20 mm, walls held at 90 C
  winding         : 4 x 8 = 32 wires of 2 mm, copper fill 0.50
  bundle k_eff    : 0.70 W/m.K   (copper alone: about 400)
  current         : 31.4 A per wire = 10.0 A/mm^2   (the notebooks use 10 to 45 A)
  one wire at 90 C: R = 6.98 mOhm/m, R I^2 = 6.89 W/m
  whole slot      : 220 W per metre of slot
  heat source q   : 1.10 MW/m^3 at the wall temperature, +0.31 % per kelvin of rise
  thermal runaway : 158 A, 3.5 x the largest current used
```

![One stator slot, with half of each neighbour](https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/Ex07.1_slot.png)

**The winding is treated as one material.** Modelling every wire would make the
conductivity jump from about 400 W/m·K in the copper to about 0.2 in the enamel
and resin, at every wire edge. Machine designers average it instead: one
effective conductivity for the bundle, and the heat of the 32 wires spread
evenly over the slot. An alternating current heats like a direct one at its RMS
value, so $I$ here is the RMS current.

### The heat source

**What these cells do.** You write $q(\theta, I)$, the heat the winding makes per cubic metre when the slot is $\theta$ kelvin above the wall; the next cell checks it against `pb.heat_source` and plots the winding's heat against the current at two copper temperatures.

**How.** One metre of wire has resistance $\rho/A_{\text{wire}}$ and makes $RI^2$ of heat. Thirty-two of them, spread over the slot's cross-section, make

$$q = \text{fill}\cdot\rho(T)\,J^2, \qquad J = \frac{I}{A_{\text{wire}}}, \qquad \rho(T) = \rho_{20}\,[1 + \alpha\,(T - 20)], \qquad T = 90 + \theta,$$

with `pb.FILL`, `pb.A_WIRE`, `pb.RHO_CU`, `pb.ALPHA_CU` and `pb.T_WALL` for the constants.

In [ ]:
def heat_source(theta, I):
    J = I / pb.A_WIRE                                                # current density, A/m^2
    rho = pb.RHO_CU * (1 + pb.ALPHA_CU * (pb.T_WALL + theta - 20))   # copper at T = 90 C + theta
    return pb.FILL * rho * J ** 2                                    # 32 wires' R I^2, over the slot

In [ ]:
theta_test = np.array([0.0, 10.0, 40.0])        # kelvin above the wall
for I in (pb.I_RANGE[0], pb.I_RATED, pb.I_RANGE[1]):
    check(f"heat_source at {I:.1f} A", heat_source(theta_test, I) / pb.heat_source(theta_test, I),
          np.ones(3), tol=1e-12)

currents = np.linspace(*pb.I_RANGE, 60)
fig, ax = plt.subplots(figsize=(6.6, 4.0))
for T, col in ((pb.T_WALL, CYCLE[1]), (pb.T_WALL + 40, CYCLE[0])):
    ax.plot(currents, pb.N_WIRES * pb.heat_per_wire(currents, T), color=col, lw=2,
            label=f"copper at {T:.0f} °C")
ax.axvline(pb.I_RATED, color="0.6", ls=":", lw=1)
ax.set_xlabel("current per wire  [A]"); ax.set_ylabel("heat  [W per metre of slot]")
ax.set_title("RI² of the whole winding"); ax.legend(frameon=False)
plt.show()

**What you should see.** Three `PASS` lines, and two rising curves. Both grow
as $I^2$; the upper one is the same winding 40 K hotter and makes **12 % more
heat**, because copper's resistance has risen by that much. Inside the slot the
copper is hotter than the wall, so it makes more heat, which heats it further.
Split the heat into a fixed part and a part that grows with the rise,
$q = c_0 + c_1\theta$ — `pb.heat_coefficients(I)` returns the two. From here on
the notebook uses `pb.heat_source`, the function your version was checked against.

---

## 2 · The exact and the finite-difference solution

**What this cell does.** Computes the temperature rise at rated current two ways — exactly, and by finite differences on a first mesh of 41 × 81 nodes — and draws the two side by side with their difference on the same nodes.

**The exact solution.** Because $c_0$ and $c_1$ are constants and the slot is a rectangle held at zero, the equation $k_{\text{eff}}\nabla^2\theta + c_1\theta = -c_0$ can be solved exactly. With $x' = x + W/2$ and $y' = y + H/2$ measured from a corner, every mode of the slot,

$$u_{mn} = \sin\frac{m\pi x'}{W}\,\sin\frac{n\pi y'}{H},$$

is zero on the walls and turns into a multiple of itself under the Laplacian, $\nabla^2 u_{mn} = -\lambda_{mn}u_{mn}$ with $\lambda_{mn} = (m\pi/W)^2 + (n\pi/H)^2$. A constant is $\sum 16/(\pi^2 mn)\,u_{mn}$ over odd $m$ and $n$, so

$$\theta = \sum_{m,\,n\ \text{odd}} \frac{16\,c_0}{\pi^2\,m\,n\,\bigl(k_{\text{eff}}\lambda_{mn} - c_1\bigr)}\;u_{mn}.$$

The first term is a single dome, 21.54 K high in the middle; the smaller terms after it correct its shape. `pb.exact_solution(I, x, y)` sums them up to 399 in each direction, exact to about $10^{-5}$ K. A real slot — a rounded bottom, tooth tips, a liner — has no such formula, which is why finite differences and networks exist; here it lets both be scored exactly.

**The finite-difference solution.** A grid of 41 nodes across the 10 mm width and $2 \times 41 - 1 = 81$ along the 20 mm depth, $h = 0.25$ mm both ways. Each of the $39 \times 79 = 3081$ nodes inside is one unknown, and the five-point stencil gives it one linear equation,

$$k_{\text{eff}}\,\frac{\theta_N + \theta_S + \theta_E + \theta_W - 4\theta_P}{h^2} + c_0 + c_1\theta_P = 0 .$$

`pb.fdm_solve` builds the sparse system and solves it. Section 3 asks how fine the mesh has to be.

In [ ]:
NX = 41                                                     # a first mesh; section 3 chooses the right one
x41, y41 = pb.fdm_grid(NX)                                  # the 41 x 81 node coordinates, m
theta_exact = pb.exact_solution(pb.I_RATED, x41, y41)       # exact, on those nodes
_, _, theta_fdm = pb.fdm_solve(pb.I_RATED, NX)              # finite differences, same nodes
diff = theta_fdm - theta_exact
print(f"hot spot: exact {theta_exact.max():.2f} K, finite differences {theta_fdm.max():.2f} K")
print(f"largest difference anywhere: {np.abs(diff).max():.2e} K")
for n in (1, 3, 9, 49, 399):                                # terms of the series in each direction
    centre = pb.exact_solution(pb.I_RATED, [0.0], [0.0], n_terms=n)[0, 0]
    print(f"  exact solution with terms up to {n:3d}: hot spot {centre:6.2f} K")

fig, axes = plt.subplots(1, 3, figsize=(12.6, 5.6))
pb.plot_field(theta_exact.ravel(), NX, 2 * NX - 1, ax=axes[0], wires=True, title="exact")
pb.plot_field(theta_fdm.ravel(), NX, 2 * NX - 1, ax=axes[1], wires=True, title="finite differences, 41 x 81")
pb.plot_field(diff.ravel(), NX, 2 * NX - 1, ax=axes[2], title="finite differences minus exact",
              label="difference  [K]", cmap="coolwarm")
for ax in axes:                                             # the 41 x 81 mesh, thin white lines
    ax.vlines(x41 * 1e3, y41[0] * 1e3, y41[-1] * 1e3, colors="white", lw=0.25, alpha=0.7)
    ax.hlines(y41 * 1e3, x41[0] * 1e3, x41[-1] * 1e3, colors="white", lw=0.25, alpha=0.7)
plt.tight_layout(); plt.show()

**What you should see.** The same hot spot both ways, **18.74 K** above the
wall — 108.74 °C in the copper — and two fields you cannot tell apart. The thin
white lines are the mesh, 0.25 mm apart, drawn on all three panels. The third
panel shows the difference on the same nodes: a few thousandths of a kelvin,
finite differences everywhere slightly too cold, most in the middle, zero on the
walls. The series closes in on the answer from both sides: one dome is 21.54 K,
terms up to 3 give 18.17 K, up to 9 give 18.79 K, and from 49 on 18.74 K.

---

## 3 · Optimising the mesh density

**What this cell does.** Solves the slot on seven meshes, from 5 to 161 nodes across, at rated current and at 45 A, the largest current the set uses, times every solve, and — in the TODO below — measures the error of the hot spot against the exact solution.

**The agreed accuracy.** The hot spot is the number a machine designer uses: it is compared with the insulation's class limit, which is set in whole kelvin. We agree that **0.01 K** on the hot spot is accurate enough, at every current in the set — the second decimal the notebook prints. The mesh to use is the coarsest that meets it, because every refinement costs time.

In [ ]:
GRIDS = [5, 9, 11, 21, 41, 81, 161]                          # nodes across; odd, so the centre is a node
sols, times = {}, {}
for nx in GRIDS:
    for I in (pb.I_RATED, 45.0):
        pb.fdm_solve(I, nx)                                  # warm-up, not timed
        t0 = time.perf_counter()
        _, _, th = pb.fdm_solve(I, nx)                       # grid + matrix + solve
        sols[nx, I] = th
    times[nx] = time.perf_counter() - t0
print("solved on", ", ".join(f"{g} x {2*g-1}" for g in GRIDS), "at", f"{pb.I_RATED:.2f} A and 45 A")

In [ ]:
hot_err = {}
for (nx, I), th in sols.items():
    exact_hot = pb.exact_solution(I, [0.0], [0.0])[0, 0]      # the exact hot spot, at the centre
    hot_err[nx, I] = abs(th.max() - exact_hot)               # the error of the hot spot, K

In [ ]:
TARGET = 0.01                                               # K: the agreed accuracy of the hot spot
hs = np.array([10.0 / (g - 1) for g in GRIDS])              # node spacing, mm
worst = np.array([max(hot_err[g, pb.I_RATED], hot_err[g, 45.0]) for g in GRIDS])
NX = next(g for g, e in zip(GRIDS, worst) if e <= TARGET)   # the coarsest mesh that meets it
for g, h, e in zip(GRIDS, hs, worst):
    print(f"  {g:4d} x {2*g-1:4d}  h = {h:4.2f} mm  hot-spot error {e:.2e} K  {times[g]*1e3:6.1f} ms"
          + ("   <- the optimum" if g == NX else ""))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.0, 4.2))
for I, col in ((pb.I_RATED, CYCLE[1]), (45.0, CYCLE[0])):
    a1.plot(hs, [hot_err[g, I] for g in GRIDS], "o-", color=col, lw=2, label=f"{I:.2f} A")
a1.axhline(TARGET, color=CYCLE[3], ls="--", lw=1.2, label="agreed accuracy, 0.01 K")
a1.set_xlabel("node spacing $h$  [mm]"); a1.set_ylabel("hot-spot error  [K]")
a1.set_title("the hot spot's error as the mesh is refined"); a1.legend(frameon=False)
a2.plot(hs, [times[g] * 1e3 for g in GRIDS], "o-", color=CYCLE[4], lw=2)
a2.set_xlabel("node spacing $h$  [mm]"); a2.set_ylabel("time per solve  [ms]")
a2.set_title("and what it costs")
k = GRIDS.index(NX)
a1.plot(hs[k], worst[k], "o", ms=13, mfc="none", mec=CYCLE[3], mew=2)   # the optimum
a2.plot(hs[k], times[NX] * 1e3, "o", ms=13, mfc="none", mec=CYCLE[3], mew=2)
plt.tight_layout(); plt.show()
x_opt, y_opt = pb.fdm_grid(NX)                              # the optimum mesh's nodes

**What you should see.** On the left, the hot-spot error falling fast as $h$
shrinks — it is proportional to $h^2$, so halving $h$ divides it by four — and
larger at 45 A than at rated current, because the hot spot itself is twice as
high. On the right, the time: flat at a couple of milliseconds for coarse
meshes, then climbing steeply once the unknowns run into the tens of thousands.

**The optimum is 41 × 81 nodes, $h = 0.25$ mm** — the mesh of section 2.
21 × 41 misses 0.01 K at 45 A ($1.9\times10^{-2}$ K); 41 × 81 meets it at both
currents ($4.9\times10^{-3}$ K at 45 A) in under 10 ms, with 3,081 unknowns.
Every finer mesh is more accurate than anyone asked for: the 161 × 321 mesh —
51,681 nodes — takes about thirty times longer to give $3\times10^{-4}$ K, which
the agreed 0.01 K does not need. **The accuracy asked for decides the mesh**,
not the other way round.

---

## 4 · A physics-informed network

**What this cell does.** Builds the network, chooses its collocation points, and prints its size against the number of points it trains on.

**The inputs.** The slot is $2a$ wide and $2b$ deep: $a = 5$ mm and $b = 10$ mm are its half-width and half-depth. The network sees three numbers, each running from −1 to 1:

$$\xi = \frac{x}{a}, \qquad \eta = \frac{y}{b}, \qquad s = 2\,\frac{I - 10\ \text{A}}{45\ \text{A} - 10\ \text{A}} - 1,$$

the position across the slot, the position along it, and the current. With the current as an input, **one training answers every current** from 10 to 45 A.

**The output.** L7.1's **Solution Ansatz** slide writes the network as the solution, $u \approx \mathcal{N}$. Here the network's output is multiplied by a known size, so that it only has to learn the shape:

$$\hat\theta(\xi, \eta, s) = \Theta\left(\frac{I}{I_{\text{rated}}}\right)^{2} \mathcal{N}(\xi, \eta, s), \qquad \Theta = 20\ \text{K},$$

$\Theta$ being about the hot spot at rated current and $(I/I_{\text{rated}})^2$ how the heat grows with the current; K is the kelvin. Two ways to impose the walls:

* **soft enforcement** (L7.1's **Soft Enforcement** slide) — $\hat\theta$ as it stands, with a penalty in the loss at points on the walls;
* **hard enforcement** (L7.1's **Hard Enforcement** slide, $u = A + D\,\mathcal{N}$ with $A = 0$ here) — $\hat\theta$ multiplied by the mask $D = (1 - \xi^2)(1 - \eta^2)$, which is zero on every wall.

**Why 2000 collocation points.** They are not tied to the finite-difference mesh. A mesh carries one unknown per node, for one current; the network's unknowns are its weights, and the collocation points are where its residual is checked — spread over the slot **and** the current range together. L7.1's **Solution Ansatz** slide asks for at least $N^*$ of them, a number set by the neurons; four layers of 32 give $N^* = 136$, and 2000 is about fifteen times that, drawn by Latin hypercube so they cover the three inputs evenly. More points cost training time without improving the answer here; fewer than $N^*$ let the network satisfy the equation at the points and do anything between them. The soft penalty uses 400 more on the walls. The points stay fixed, because L-BFGS needs the same points at every step.

In [ ]:
a, b = pb.A_HALF, pb.B_HALF
I_LO, I_HI = pb.I_RANGE
THETA = 20.0                                                # K: the size of the answer, Theta
Q = pb.heat_coefficients(pb.I_RATED)[0]                     # W/m^3: the size of the heat

def current(s):
    return I_LO + (s + 1) / 2 * (I_HI - I_LO)               # s in -1..1 back to amperes

class Slot(torch.nn.Module):                                # a network: layers in __init__, the computation in forward
    def __init__(self, hard, width=32, layers=4):
        super().__init__()
        self.hard = hard
        self.net = MLP(n_in=3, n_hidden=width, n_layers=layers)   # dense tanh network: xi, eta, s in

    def forward(self, p):
        xi, eta, s = p[:, :1], p[:, 1:2], p[:, 2:]
        out = THETA * (current(s) / pb.I_RATED) ** 2 * self.net(p)
        return out * (1 - xi ** 2) * (1 - eta ** 2) if self.hard else out   # the mask, for hard enforcement

box = ((-1, 1), (-1, 1), (-1, 1))                           # xi, eta and the current
inner = to_tensor(interior_points(2000, box, method="lhs", seed=1), requires_grad=True)  # collocation points, differentiable
w2 = boundary_points(100, ((-1, 1), (-1, 1)), seed=1)       # 400 points on the four walls
walls = to_tensor(np.hstack([w2, np.random.default_rng(1).uniform(-1, 1, (len(w2), 1))]))  # each at a random current

describe(Slot(hard=True).net, n_collocation=len(inner))  # print the parameter count and the collocation-to-parameter ratio
fig, ax = plt.subplots(figsize=(4.2, 6.0))
pin = to_numpy(inner)
ax.plot(pin[:, 0] * a * 1e3, pin[:, 1] * b * 1e3, ".", ms=2.5, color=CYCLE[1], label="inside: the residual")
ax.plot(w2[:, 0] * a * 1e3, w2[:, 1] * b * 1e3, ".", ms=4, color=CYCLE[0], label="walls: the soft penalty")
ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); ax.set_ylabel("y  [mm]")
ax.set_title("collocation points, all currents together")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1))
plt.show()

**What you should see.** The network's size and its collocation ratio, and the
slot filled with blue points and ringed in orange. Unlike the mesh of section
2 the points follow no grid: they are drawn at random, spread evenly by the
Latin hypercube, and each one also carries its own current.

### The residual

**What this cell does.** You write the residual: the slot's equation with the network in place of $\theta$, divided by the rated heat $Q$ so that it is a number of order one.

**How.** `d2(th, p, 0)` is the second derivative in $\xi$. Since $\xi = x/a$, $\partial^2/\partial x^2 = (1/a^2)\,\partial^2/\partial\xi^2$, and likewise in $\eta$ with $b$. The heat is `pb.heat_source` at the current `current(p[:, 2:])`.

In [ ]:
def residual(model, p):
    th = model(p)                                                     # the network's rise, K
    lap = d2(th, p, 0) / a ** 2 + d2(th, p, 1) / b ** 2               # the Laplacian by autograd, in metres
    return (pb.K_EFF * lap + pb.heat_source(th, current(p[:, 2:]))) / Q   # the equation, scaled

### Training, with soft and with hard enforcement

**What this cell does.** Trains the same network twice, on the same points with the same budget: once with soft enforcement of the walls, once with hard.

**How.** The soft loss is the mean squared residual plus the mean squared wall rise, $\mathcal{L} = \overline{r^2} + w\,\overline{(\hat\theta_{\text{wall}}/\Theta)^2}$ with $w = 1$; the hard loss is the residual alone, because the mask has already met the walls. Both use the course's schedule: 1500 steps of Adam at a learning rate of $2\times10^{-3}$, then 150 steps of L-BFGS. About a minute each on a CPU.

In [ ]:
nets, train_time = {}, {}
for name, hard in (("soft", False), ("hard", True)):
    set_seed(88)  # fix the random start so a run repeats
    model = Slot(hard).to(DEVICE)                            # four layers of 32

    def loss():                                              # what train_two_stage minimises
        l = residual(model, inner).pow(2).mean()             # the equation, inside
        if not hard:
            l = l + (model(walls) / THETA).pow(2).mean()         # the soft wall penalty, w = 1
        return l

    t0 = time.perf_counter()
    train_two_stage(model, loss, adam_steps=1500, lbfgs_steps=150, lr=2e-3, report_every=0)  # Adam first, then L-BFGS
    nets[name], train_time[name] = model, time.perf_counter() - t0
    print(f"{name} enforcement: trained in {train_time[name]:.0f} s")

**What you should see.** Two lines, each a minute or so.

---

## 5 · The three answers compared

**What this cell does.** Scores the three methods against the exact solution at six currents from 10 to 45 A: finite differences on the optimum mesh of section 3, and the two networks on a fine grid of 161 × 321 points. It prints the worst hot-spot error and the worst error anywhere, the time one current takes and the training time, then maps each method's error at 45 A.

**How.** Finite differences only give values at their own nodes, so they are scored there; a network gives a value at any point, so it is scored on the fine grid. The maps show the error with its sign: blue where the method is too cold, red where it is too hot, white where it is right.

In [ ]:
TEST_I = np.array([10.0, 17.5, 25.0, pb.I_RATED, 38.0, 45.0])
xf, yf = pb.fdm_grid(161)                                    # a fine grid for the networks
Pf = np.stack([m.ravel() for m in np.meshgrid(xf, yf)], axis=1)

def evaluate(model, I):
    s = 2 * (I - I_LO) / (I_HI - I_LO) - 1
    p = to_tensor(np.hstack([Pf / [a, b], np.full((len(Pf), 1), s)]))  # the network's inputs on the fine grid
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy(model(p)).reshape(len(yf), len(xf))

def middle(fn, reps=5):
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

methods = {f"finite differences, {NX} x {2*NX-1}": (lambda I: pb.fdm_solve(I, NX)[2], x_opt, y_opt),
           "PINN, soft enforcement": (lambda I: evaluate(nets["soft"], I), xf, yf),
           "PINN, hard enforcement": (lambda I: evaluate(nets["hard"], I), xf, yf)}
hot, worst, per_current = {}, {}, {}
for name, (f, x, y) in methods.items():
    errs = [(abs(f(I).max() - pb.exact_solution(I, [0.0], [0.0])[0, 0]),
             np.abs(f(I) - pb.exact_solution(I, x, y)).max()) for I in TEST_I]
    hot[name], worst[name] = max(e[0] for e in errs), max(e[1] for e in errs)
    per_current[name] = middle(lambda: f(28.0))
names = list(methods)
train = dict(zip(names, (0.0, train_time["soft"], train_time["hard"])))

print(f"{'':30s}{'hot spot':>12s}{'anywhere':>12s}{'per current':>13s}{'training':>10s}")
for n in names:
    print(f"  {n:28s}{hot[n]:10.2e} K{worst[n]:10.2e} K{per_current[n]*1e3:10.2f} ms{train[n]:8.0f} s")

fig, axes = plt.subplots(1, 3, figsize=(12.0, 5.6))
for ax, (name, (f, x, y)) in zip(axes, methods.items()):
    pb.plot_field((f(45.0) - pb.exact_solution(45.0, x, y)).ravel(), len(x), len(y), ax=ax,
                  title=f"{name}\nminus exact, 45 A", label="error  [K]", cmap="coolwarm")
plt.tight_layout()
plt.savefig(cc.output_path("Ex07.1_report.png"), dpi=150, bbox_inches="tight")
plt.show()

**What you should see.** Finite differences on the 41 × 81 mesh: about
$5\times10^{-3}$ K on the hot spot and $7\times10^{-3}$ K at worst — within the
agreed 0.01 K — in about 7 ms per current. The network with **hard
enforcement**: about **0.02 K**, anywhere and at every current, exactly zero on
the walls — close to the agreed accuracy, but not within it. The network with
**soft enforcement**: about 0.01 K on the hot spot, but about **1.5 K** off
elsewhere, worst at the walls, where the condition was only asked for. A
network answers a current in about 10 ms here, evaluated on all 51,681 points
of the fine grid.

**What the comparison says.**

* **Hard enforcement is the better of the two networks**, by almost a hundred
  times. A soft wall is a request: the loss trades it against the equation, the
  wall ends up warm, and because the walls set the temperature inside, the
  error spreads through the whole slot. The mask never offers that trade.
* **Finite differences meet the agreed accuracy for almost nothing**: a few
  milliseconds per current and no training. Neither network reaches 0.01 K with
  this training budget; the hard-enforcement network comes within a factor of
  two, after a minute of training.
* **The network's advantage is that one training answers every current**, and a
  mesh is not needed. Where a mesh is costly — awkward 3-D shapes, unknown
  parameters found from data, many inputs at once — the balance moves towards
  the network, and that is where the rest of Part 2 goes.

---

## 6 · Your report

**What these cells do.** You answer four questions and the question that concludes the set; the next cell writes `Ex07.1_report.md` with the results table, the error maps of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L7.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers.

**How to answer.** In the cell below, write your name in `NAME`, and each answer between the pair of triple quotes `` that follows its question. Leave the questions themselves as they are. An answer needs at least eight words; the cell after it says which answers are still missing or too short.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Finite differences turned the slot into one linear system. What are its unknowns, "
    "how many did the optimum mesh have, and what takes the place of the mesh in a "
    "physics-informed network? (-> L7.1 Q4)": """
""",
    "Which mesh meets the agreed accuracy of 0.01 K, and how did the hot-spot error "
    "change each time the spacing h was halved? (-> L7.1 Q4)": """
""",
    "Why is a wall imposed by soft enforcement met only approximately, and why does "
    "its error not stay on the wall? (-> L7.1 Q7)": """
""",
    "What does the mask of hard enforcement guarantee, and what would it cost for a "
    "slot with a rounded bottom? (-> L7.1 Q9)": """
""",
    "To conclude: finite differences met the agreed 0.01 K for almost nothing. For which problem would you train "
    "a physics-informed network instead, and what would you check it against? "
    "(-> L7.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1)
         if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | hot-spot error | worst error | per current | training |", "|---|---|---|---|---|"]
    rows += [f"| {n} | {hot[n]:.2e} K | {worst[n]:.2e} K | {per_current[n]*1e3:.2f} ms | {train[n]:.0f} s |" for n in names]
    out = ["# Ex_07.1 — PINN Thermal Prediction", "", f"**{NAME}**", "",
           "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", f"Exact hot spot at rated current: {theta_exact.max():.2f} K above the wall.", "",
           *rows, ""]
    for q, text in ANSWERS.items():
        out += [f"## {q}", "", text.strip(), ""]
    path = cc.output_path("Ex07.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready`, with the answers still missing or
shorter than eight words, until every answer is yours; then the path of the
report. Then run the cell below: it turns the report into a PDF,
with the error maps of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex07.1_report.md into Ex07.1_report.pdf, with any figure
# saved as Ex07.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex07.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex07.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex07.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex07.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 7 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**: the course builds it with a classical
solver once you have chosen the project, and shows that it has converged. You
do not solve the physics a second way; the month goes into the deep learning —
making a network predict what the ground truth says, and showing how well it
does.

### Mini project A · The slot with its insulation

**The problem.** Ex_07.1 averaged the winding into one material. Here every one
of the 32 wires is resolved: a 2 mm copper core, its enamel coat, and the
impregnating resin that fills the gaps between the wires. The enamel is thin:
a grade 2 wire, the usual choice for a motor, is at least 0.075 mm thicker than
its 2 mm conductor (IEC 60317), so the coat is about 0.04 mm — a fiftieth of
the wire. Copper conducts about 400 W/m·K, the enamel and an unfilled resin
about 0.2, so the conductivity drops some two-thousandfold at every copper
edge. The heat is made in the copper only.

**Why a plain PINN is not enough.** The temperature is continuous at every
edge but its slope is not — the heat flow $k\,\partial\theta/\partial n$ is what
stays the same on both sides. A smooth network rounds that corner off, and a
coat 0.04 mm thick in a 10 mm slot is so thin that random collocation points
hardly ever land in it.

**What you build.** A domain-decomposed PINN: one network per material, joined
by interface terms in the loss that ask for equal temperature and equal heat
flow across every edge. You choose how to sample the thin layers, how to weight
the interface terms, and whether the walls use soft or hard enforcement.

**The ground truth you get.** The resolved slot solved by finite differences,
with several nodes across the enamel and a conductivity set cell by cell, for
the currents of Ex_07.1, with its grid-refinement table.

**What you hand in.** Error maps against the ground truth, in each material;
the hot spot and its error; how well the heat flow matches across the edges;
and how far the averaged slot of Ex_07.1 was from the resolved one.

### Mini project B · How old is the insulation?

**The problem.** Insulation ages. Heat degrades the enamel and the resin, the
resin comes away from the wires and voids open. A void holds air, which
conducts about a tenth as well as the resin it replaces, so the conductivity
falls where that happens — the copper runs hotter, which ages the insulation
faster. Nobody can open a machine in service to look. A few temperature sensors
in the slot, read at a few currents, are all there is.

**What you build.** An inverse PINN on the averaged slot of Ex_07.1, now with a
conductivity that varies over the slot: one network for the temperature
$\theta(x, y, I)$ and one for the conductivity $k(x, y)$, kept positive, fitted
together to the heat equation $\nabla\cdot(k\nabla\theta) + q = 0$ and to the
sensor readings.

**The ground truth you get.** Slots with a degraded region hidden in them,
solved by finite differences: the sensor readings, with their positions,
currents and noise level, and the true conductivity and temperature fields to
score yourself against afterwards.

**What you hand in.** The recovered conductivity against the true one, the
predicted hot spot against the true one, and how both depend on the number of
sensors and the noise.